In [0]:
#%run ./transform_data ----- A décommmenter pour lancer les notebooks séparements

In [0]:
#pour calculer la date de fin de prd 
#Si kiln_unload_end_date est remplie alors la prendre
#Sinon si kiln_unload_start_date est remplie alors la prendre
#Sinon si germ_unload_start_date est remplie alors la prendre + 2 jours
#Sinon si steep_c1_o1_filling_start_date est remplie alors la prendre + 8 jours
#Sinon planned_date + 8 jours

date_fin_prd = batches_info.alias("a").join(
    localisation.alias("b"),
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left"
).withColumn(
    "date_fin_prod",
    when(F.col("kiln_unload_end_date").isNotNull(), F.col("kiln_unload_end_date"))
    .when(F.col("kiln_unload_start_date").isNotNull(), F.col("kiln_unload_start_date"))
    .when(F.col("germ_unload_start_date").isNotNull(), F.expr("germ_unload_start_date + interval 2 days"))
    .when(F.col("steep_c1_o1_filling_start_date").isNotNull(), F.expr("steep_c1_o1_filling_start_date + interval 8 days"))
    .otherwise(F.expr("a.planned_datetime + interval 8 days"))
).select(
    "a.*",
    "date_fin_prod"
)



In [0]:
batch_status = date_fin_prd.alias("a").join(
    batches_status.alias("b"),
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left"
).select(
    "a.*",
    "b.batch_status_localization",
    "b.error_messages")


In [0]:
batch_status_select = batch_status.select(
    "batch_id",
    "batch_number",
    "mes_number",
    "production_type",
    "requirement_specifications",
    "specy_name",
    "variety_name",
    # Phase 2 : identifiants de traduction, propagés depuis batches_info
    "id_good_specy",
    "id_good_variety",
    "id_parameter_production_type",
    "id_requirement_specification",

    "cycle_duration",
    "cycle_label",

    "goods_weight",
    "planned_date",
    "planned_datetime",
    "date_fin_prod",
    "batch_status_localization",
    "error_messages")

In [0]:
df_with_week_year = add_date_columns(batch_status_select, "date_fin_prod")
table_batches_specifications = df_with_week_year.dropDuplicates()

In [0]:
table_batches_specifications = table_batches_specifications \
    .withColumn("goods_weight", F.col("goods_weight").cast("integer"))

Merge fonction delta

In [0]:
current_process= "dim_batches_specifications"

In [0]:
target_table_batches_specifications = current_catalog +"."+current_schema+"."+current_process
print(target_table_batches_specifications)

In [0]:
all_columns =  table_batches_specifications.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = ['batch_id']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    table_batches_specifications, 
    target_table_batches_specifications, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )